# Reusable assets

Teams rarely build every workflow from scratch. The SDK exposes three
kinds of **reusable assets** you can share across workflows:

| Asset | Client resource | What it holds |
|---|---|---|
| **Node libraries** | `client.node_libraries` | reusable node/edge bundles (a `BaseNodeConfig`) |
| **Templates** | `client.templates` | a reference to a reusable workflow (`workflow_id` + access level) |
| **Categories** | `client.categories` | the team's workflow category names (read-only) |

This notebook walks all three: schema discovery + full CRUD for node
libraries and templates, and listing categories.

> A **template** is a pointer to an existing workflow — its config carries a
> `workflow_id` and an `access_level`, and its display name/description/category
> come from the referenced workflow. So you first build (or already have) a
> workflow, then "save it as a template".

See the companion guide: [`../docs/guides/reusable_assets.md`](../docs/guides/reusable_assets.md).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

# Reads INTERACTLY_API_KEY / INTERACTLY_TEAM_ID / INTERACTLY_USER_ID /
# INTERACTLY_BASE_URL from the environment.
client = AsyncWorkflowClient()
print("Connected to", client._base_url)

In [ ]:
# Idempotency guard — remove any assets left over from a previous run so that
# re-running this notebook always starts from a clean slate.
_DEMO_LIB_NAMES = {"Standard Greeting", "Standard Greeting v2"}
_DEMO_WF_NAME = "Insurance ChatBot Starter (11_reusable_assets)"


def _lib_name(cfg):
    return cfg.get("name") if isinstance(cfg, dict) else getattr(cfg, "name", None)


# Node libraries are matched by their config name.
for _lib in await (await client.node_libraries.list(size=200)).list_all():
    if _lib_name(_lib.node_library_config) in _DEMO_LIB_NAMES:
        await client.node_libraries.delete(_lib.id)

# Deleting the demo workflow also cascades to any template that references it.
for _wf in await (await client.workflows.list(size=200)).list_all():
    if _wf.name == _DEMO_WF_NAME:
        await client.workflows.delete(_wf.id)

print("Cleared any pre-existing demo assets.")

## 1 · Node libraries

A node library wraps a single `node_library_config` (a
`BaseNodeConfig`-compatible dict or typed config object). Start by
fetching the JSON Schema so you know what shape the server expects.

In [ ]:
node_lib_schema = await client.node_libraries.schema()
# The schema dict describes the accepted config; peek at its top-level keys.
print(sorted(node_lib_schema.keys()))

### Create a node library

`create()` takes a keyword-only `node_library_config`. Here we pass a
plain dict, but you can also pass a typed `SayStaticMessageNodeConfig`
(or any `BaseNodeConfig`) from `interactly.configs` —
the SDK serialises it for you.

In [ ]:
from interactly.types.node_libraries.node_library import NodeLibrary

library: NodeLibrary = await client.node_libraries.create(
    node_library_config={
        "node_type": "say_static_message",
        "name": "Standard Greeting",
        "description": "Reusable friendly greeting node",
        "static_messages_config": {
            "static_messages": ["Hi there! Thanks for reaching out. How can I help?"]
        },
    }
)
print(library.id)
print(library.node_library_config)

### List, get, update, delete

`list()` is paginated and supports `search` (fuzzy name) and
`access_level` filters. `update()` replaces the stored config.

In [ ]:
# List (page 1) and iterate the current page's items.
page = await client.node_libraries.list(size=10, search="Greeting")
for lib in page.items:
    print(lib.id, lib.node_library_config)

# Fetch a single library by id.
fetched = await client.node_libraries.get(library.id)

# Update its config (send the full replacement config).
updated = await client.node_libraries.update(
    library.id,
    node_library_config={
        "node_type": "say_static_message",
        "name": "Standard Greeting v2",
        "static_messages_config": {"static_messages": ["Hello and welcome!"]},
    },
)
print(updated.node_library_config)

## 2 · Templates

A template is a **saved reference to an existing workflow**. Its
`workflow_template_config` holds just two things you set:

- `workflow_id` — the workflow this template points at
- `access_level` — `"personal"`, `"team"`, or `"system"` (who can use it)

The template's display name, description, and category are **not** stored
on the template — they come from the referenced workflow. So the flow is:
**build a workflow → save it as a template**. `create()` / `update()` take
a plain `config` dict (a `WorkflowTemplateConfig`-compatible payload); on
`update()` only `workflow_id` and `access_level` are mutable.

In [ ]:
from interactly.configs import (
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)

# The config schema describes what the server accepts (workflow_id + access control).
template_schema = await client.templates.schema()
print(sorted(template_schema.keys()))

# 1. Build the workflow we want to make reusable (a minimal one-node greeter).
workflow = await client.workflows.create_from_config(
    WorkflowConfigFullyHydrated(
        workflow_config=WorkflowConfig(
            name="Insurance ChatBot Starter (11_reusable_assets)",
            description="A minimal greeting workflow",
            category="System Examples",
        ),
        node_configs=[
            SayStaticMessageNodeConfig(
                name="Greeting",
                is_start=True,
                static_messages_config=StaticMessagesConfig(
                    static_messages=["Hi there! Thanks for reaching out. How can I help?"]
                ),
            )
        ],
        edge_configs=[],
    )
)
print("workflow:", workflow.id, workflow.name)

# 2. Save it as a team template — the config just points at the workflow.
template = await client.templates.create(
    config={"workflow_id": str(workflow.id), "access_level": "team"}
)
print("template:", template.id)
print(template.workflow_template_config)

In [ ]:
# List / get / update / delete mirror the node-libraries API. A template row
# carries the workflow_id it points at (its name comes from that workflow).
template_page = await client.templates.list(size=10)
for t in template_page.items:
    print(t.id, "→ workflow_id:", (t.workflow_template_config or {}).get("workflow_id"))

one = await client.templates.get(template.id)
print("fetched:", one.id)

# Update a mutable field — here we narrow the template from team- to personal-scope.
one = await client.templates.update(template.id, config={"access_level": "personal"})
print("after update:", one.workflow_template_config)

## 3 · Categories

Categories are read-only: `list()` returns the team's category names —
global defaults merged with any team-specific custom categories
(deduplicated, globals first). Use one of these strings as the
`category` field on a workflow or template.

In [ ]:
categories = await client.categories.list()
print(categories)

## Cleanup

Delete the assets we created so we don't leave test data behind.

In [ ]:
await client.node_libraries.delete(library.id)
await client.templates.delete(template.id)
# Also delete the workflow we created to back the template.
await client.workflows.delete(workflow.id)
print("cleaned up")

await client.close()

## See also

- Guide: [`../docs/guides/reusable_assets.md`](../docs/guides/reusable_assets.md)
- [`10_llm_configs.ipynb`](10_llm_configs.ipynb) — another shareable team asset
- [`02_interactive_workflow.ipynb`](02_interactive_workflow.ipynb) — building the workflow a template points at